### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [1]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown
from autogen_ext.models.openai._model_info import ModelInfo, ModelFamily

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

In [2]:
model_info = ModelInfo(
            family=ModelFamily.GPT_4O,  # Required parameter (GPT was invalid)
            vision=True,
            function_calling=True,
            json_output=True,
            structured_output=True
        )

### Start with our Message class

In [3]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [4]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [5]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [6]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [7]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        # model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        model_client = OpenAIChatCompletionClient(
            model="openai/gpt-oss-20b",
            base_url="http://localhost:1234/v1",
            model_info=model_info
        )
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        # model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        model_client = OpenAIChatCompletionClient(
            model="openai/gpt-oss-20b",
            base_url="http://localhost:1234/v1",
            model_info=model_info
        )
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        # model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        model_client = OpenAIChatCompletionClient(
            model="openai/gpt-oss-20b",
            base_url="http://localhost:1234/v1",
            model_info=model_info
        )
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [8]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [9]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [10]:
display(Markdown(response.content))

## Pros of AutoGen:
**Pros of Using AutoGen for an AI‑Agent Project**

| # | Advantage | Why it matters |
|---|-----------|----------------|
| 1 | **Modular, plug‑and‑play architecture** | Agents, tools, and prompts are independent modules that can be swapped or upgraded without touching the core logic. This speeds iteration and reduces technical debt. |
| 2 | **Built‑in agent orchestration** | AutoGen manages dialogue loops, memory, and context switching automatically, so you spend less time wiring agents together and more on domain logic. |
| 3 | **Extensible tool integration** | It natively supports adding external APIs (e.g., databases, web services) as “tools” that agents can call, enabling richer capabilities with minimal boilerplate. |
| 4 | **Observability & debugging utilities** | The framework logs each turn, tracks state changes, and provides visual dashboards, making it easier to diagnose failures or optimize agent behavior. |
| 5 | **Open‑source community & rapid evolution** | AutoGen is actively maintained on GitHub with frequent releases, documentation, and a growing ecosystem of pre‑built agents (e.g., for summarization, translation). |
| 6 | **Cross‑LLM compatibility** | It works with any LLM that exposes an API (OpenAI, Anthropic, Gemini, local models), giving flexibility to switch providers or run locally. |
| 7 | **Safety & prompt‑engineering helpers** | Built‑in safety checks and prompt templates help reduce hallucinations and enforce compliance rules out of the box. |
| 8 | **Scalable deployment patterns** | Supports containerization (Docker, Kubernetes) and serverless execution, making it straightforward to scale agents in production environments. |
| 9 | **Strong typing & error handling** | Using Python’s type hints and structured messages reduces runtime errors and improves developer confidence when composing complex workflows. |
|10 | **Rapid prototyping** | Quick start templates and example notebooks let teams prototype multi‑agent pipelines within hours, not days. |

**Bottom line:** AutoGen offers a production‑ready, highly extensible foundation that cuts development time, eases maintenance, and gives you the tools to build robust, scalable AI agent systems.

## Cons of AutoGen:
**Cons / Reasons Against Choosing AutoGen**

| Category | Limitation | Why It Matters |
|----------|------------|----------------|
| **Complexity & Learning Curve** | AutoGen is designed for advanced use cases and requires a solid grasp of Python, LLM APIs, and agent‑orchestration concepts. | New teams or those with limited coding experience may struggle to set up even simple agents, leading to longer onboarding times and higher error rates. |
| **Lack of Visual / Low‑Code Interface** | The framework is purely code‑driven; there’s no drag‑and‑drop builder or GUI for designing workflows. | Non‑technical stakeholders cannot easily modify or monitor agent logic without touching code, which can slow iteration cycles. |
| **Limited Built‑In Integrations** | AutoGen focuses on LLM interactions; it has minimal native connectors to external APIs, databases, or business tools. | Integrating with SaaS platforms (CRM, ERP, etc.) often requires writing custom adapters, adding development overhead. |
| **Scalability Concerns** | Agent orchestration is performed in‑process; scaling out across multiple machines or containers isn’t baked into the core API. | Large deployments may need additional infrastructure work (e.g., message queues, service mesh) to handle load and fault tolerance. |
| **Observability & Debugging** | Logging is basic and relies on standard Python logging. No built‑in UI for tracing agent state, message history, or performance metrics. | Troubleshooting failures or optimizing agent behavior becomes more laborious, especially in production environments. |
| **Security & Governance** | AutoGen itself does not enforce fine‑grained access controls or audit trails; developers must implement these separately. | In regulated industries (finance, healthcare), this adds extra compliance burden and potential risk exposure. |
| **Community & Ecosystem Size** | While growing, the community is still smaller than LangChain or LlamaIndex. Fewer pre‑built agents, templates, and third‑party integrations are available. | Finding ready‑made solutions for common patterns (e.g., data extraction, summarization) may require more custom coding. |
| **Dependency on External LLMs** | AutoGen delegates all intelligence to external APIs; it does not ship its own inference engine. | Any downtime or rate limits of the chosen LLM provider directly affect your agent’s availability and cost. |
| **Documentation & Examples** | Some advanced features (e.g., dynamic tool selection, custom memory stores) are documented but lack comprehensive tutorials or sample projects. | Developers may need to experiment more before achieving production‑ready behavior. |

---

### Bottom Line
AutoGen excels when you need maximum flexibility, fine‑grained control over agent logic, and the ability to prototype novel LLM‑powered workflows. However, if your team values rapid development, visual workflow design, built‑in integrations, robust observability, or strong out‑of‑the‑box scalability, AutoGen may impose unnecessary friction and risk. In such cases, consider alternatives like LangChain, CrewAI, or low‑code platforms (n8n, Zapier) that trade some flexibility for ease of use and operational robustness.



## Decision:

**Decision:** Proceed with **AutoGen** for the AI‑agent project.

**Rationale (brief):**

- The modular, plug‑and‑play architecture and built‑in orchestration give us the agility to iterate quickly on agent logic without touching core code—exactly what our prototype phase demands.
- Cross‑LLM compatibility and extensible tool integration provide the flexibility we’ll need once we move beyond a single provider or add external services (databases, APIs).
- Strong typing, error handling, and rapid prototyping templates reduce developer friction, outweighing the learning curve for an experienced Python team.
- The observability tools and open‑source community ensure that we can troubleshoot early failures and stay current with new features.

**Mitigation Plan:**

1. **Onboarding:** Allocate a half‑day workshop to cover AutoGen basics and the agent‑tool pattern.  
2. **Observability Enhancements:** Integrate an external logging/monitoring stack (e.g., Loki + Grafana) to compensate for limited built‑in UI.  
3. **Security & Governance:** Add a thin wrapper around the agent calls that logs request/response metadata and enforces role‑based access before passing data to the LLM.  

With these measures in place, AutoGen’s strengths will dominate, giving us the most scalable, maintainable foundation for our AI‑agent system.

**TERMINATE**

In [11]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [12]:
await host.stop()